# SAM-Audio - Streamlined Colab

Optimized notebook for separating an audio source with SAM-Audio using a text prompt.

Recommended flow: run the cells from top to bottom. The more fragile advanced settings (`predict_spans` and `reranking_candidates`) are disabled to keep memory usage low. The default small model is the safest option for Colab T4; use base or large on A100/L4-class runtimes.

**Requirements:**
- Colab runtime with GPU (`Runtime > Change runtime type > GPU`; A100/L4 recommended for base or large)
- Hugging Face token with approved access to the selected SAM-Audio checkpoint

## 1. Setup

In [ ]:
#@title 1. Set up SAM-Audio (run once)
import os
import pathlib
import re
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/facebookresearch/sam-audio.git"
REPO_DIR = pathlib.Path("/content/sam-audio")
FRESH_CLONE = True  #@param {type:"boolean"}
STEP_TIMEOUT_SECONDS = 1200

os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_NO_INPUT"] = "1"
# Colab includes TensorFlow, but this notebook only uses PyTorch.
# Forcing Transformers to skip TensorFlow avoids protobuf/TensorFlow import conflicts.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"


def log(message):
    now = time.strftime("%H:%M:%S")
    print(f"[{now}] {message}", flush=True)


def run_step(title, command, timeout=STEP_TIMEOUT_SECONDS):
    log(title)
    log("Command: " + " ".join(map(str, command)))
    start = time.time()
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        while True:
            line = process.stdout.readline()
            if line:
                print(line.rstrip(), flush=True)
            elif process.poll() is not None:
                break
            elif time.time() - start > timeout:
                process.kill()
                raise TimeoutError(f"Step timed out after {timeout}s: {title}")
            else:
                time.sleep(0.2)
        return_code = process.wait()
    finally:
        if process.stdout:
            process.stdout.close()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    log(f"Done: {title}")


def patch_sam_audio_for_colab(repo_dir):
    log("Applying lightweight text-only Colab patches")

    # Compatibility with recent huggingface_hub versions, plus lower-RAM checkpoint loading.
    base_file = repo_dir / "sam_audio" / "model" / "base.py"
    source = base_file.read_text(encoding="utf-8")
    source = source.replace("proxies: Optional[Dict],", "proxies: Optional[Dict] = None,")
    source = source.replace("resume_download: bool,", "resume_download: bool = False,")
    source = re.sub(r"\n\s+proxies=proxies,", "", source)
    source = re.sub(r"\n\s+resume_download=resume_download,", "", source)
    if re.search(r"\bproxies=proxies\b|\bresume_download=resume_download\b", source):
        raise RuntimeError("Hugging Face patch was not applied: check sam_audio/model/base.py")

    if "import gc\n" not in source:
        source = source.replace("import json\nimport os\n", "import gc\nimport json\nimport os\n", 1)

    original_loader = """        model = cls(config)
        state_dict = torch.load(
            os.path.join(cached_model_dir, "checkpoint.pt"),
            weights_only=True,
            map_location=map_location,
        )
        model.load_state_dict(state_dict, strict=strict)
        return model
"""
    patched_loader = """        model = cls(config)
        load_device = torch.device(map_location or "cpu")
        low_cpu_memory_load = load_device.type == "cuda" and torch.cuda.is_available()

        if low_cpu_memory_load:
            model = model.to(load_device)
            if os.environ.get("SAM_AUDIO_COLAB_HALF_ON_LOAD") == "1":
                model = model.half()
                if hasattr(model, "audio_codec"):
                    model.audio_codec = model.audio_codec.float()
            gc.collect()
            torch.cuda.empty_cache()

        state_dict = torch.load(
            os.path.join(cached_model_dir, "checkpoint.pt"),
            weights_only=True,
            map_location=load_device,
        )
        if os.environ.get("SAM_AUDIO_COLAB_TEXT_ONLY") == "1":
            optional_prefixes = (
                "vision_encoder.",
                "visual_ranker.",
                "text_ranker.",
                "span_predictor.",
            )
            for key in list(state_dict.keys()):
                if key.startswith(optional_prefixes):
                    del state_dict[key]
            gc.collect()
            if low_cpu_memory_load:
                torch.cuda.empty_cache()
        model.load_state_dict(state_dict, strict=strict)
        del state_dict
        gc.collect()
        if low_cpu_memory_load:
            torch.cuda.empty_cache()
        return model
"""
    if original_loader in source:
        source = source.replace(original_loader, patched_loader, 1)
    elif "SAM_AUDIO_COLAB_HALF_ON_LOAD" not in source:
        raise RuntimeError("Low-memory loader patch was not applied: check sam_audio/model/base.py")
    base_file.write_text(source, encoding="utf-8")

    # Avoid importing the optional judge module, which pulls perception-models.
    model_init = repo_dir / "sam_audio" / "model" / "__init__.py"
    model_init.write_text(
        "# Patched by the Colab notebook: text-only separation does not use the judge module.\n"
        "from .model import *  # noqa\n",
        encoding="utf-8",
    )

    # Avoid importing optional ranker backends (ImageBind, CLAP, judge). The notebook uses reranking_candidates=1.
    ranking_init = repo_dir / "sam_audio" / "ranking" / "__init__.py"
    ranking_init.write_text(
        "# Patched by the Colab notebook: text-only mode disables optional rankers.\n\n"
        "def create_ranker(config):\n"
        "    return None\n",
        encoding="utf-8",
    )

    # Avoid importing perception-models only for judge config type hints.
    config_file = repo_dir / "sam_audio" / "model" / "config.py"
    source = config_file.read_text(encoding="utf-8")
    source = source.replace(
        "from core.audio_visual_encoder.config import TransformerConfig as PEAVTransformerConfig\n",
        "try:\n"
        "    from core.audio_visual_encoder.config import TransformerConfig as PEAVTransformerConfig\n"
        "except Exception:\n"
        "    class PEAVTransformerConfig:\n"
        "        def __init__(self, **kwargs):\n"
        "            self.__dict__.update(kwargs)\n",
    )
    config_file.write_text(source, encoding="utf-8")

    # Disable optional video encoder and span predictor imports. Audio-only prompt separation does not use them.
    model_file = repo_dir / "sam_audio" / "model" / "model.py"
    source = model_file.read_text(encoding="utf-8")
    source = source.replace(
        "from core.audio_visual_encoder import PEAudioFrame, PEAudioFrameTransform\n",
        "class _DisabledOptionalSpanModel:\n"
        "    @classmethod\n"
        "    def from_config(cls, *args, **kwargs):\n"
        "        raise RuntimeError('Span prediction is disabled in this Colab notebook.')\n"
        "\n"
        "PEAudioFrame = _DisabledOptionalSpanModel\n"
        "PEAudioFrameTransform = _DisabledOptionalSpanModel\n",
    )
    source = source.replace(
        "from sam_audio.model.vision_encoder import PerceptionEncoder\n",
        "",
    )
    source = source.replace(
        "        self.vision_encoder = PerceptionEncoder(cfg.vision_encoder)\n",
        "        self._vision_dim = cfg.vision_encoder.dim\n"
        "        self.vision_encoder = None\n",
    )
    source = source.replace(
        "        if video is None:\n"
        "            return audio_features.new_zeros(B, self.vision_encoder.dim, T)\n"
        "        else:\n"
        "            return self.vision_encoder(video).transpose(1, 2)\n",
        "        if video is None:\n"
        "            return audio_features.new_zeros(B, self._vision_dim, T)\n"
        "        raise RuntimeError('Video-conditioned separation is disabled in this Colab notebook.')\n",
    )
    source = source.replace(
        '"(^text_encoder|^visual_ranker|^text_ranker|^span_predictor)"',
        '"(^text_encoder|^vision_encoder|^visual_ranker|^text_ranker|^span_predictor)"',
    )
    source = source.replace(
        "missing_keys = [x for x in missing_keys if not re.search(skip_regex, x)]\n"
        "            if len(missing_keys) > 0 or len(unexpected_keys) > 0:\n",
        "missing_keys = [x for x in missing_keys if not re.search(skip_regex, x)]\n"
        "            unexpected_keys = [x for x in unexpected_keys if not re.search(skip_regex, x)]\n"
        "            if len(missing_keys) > 0 or len(unexpected_keys) > 0:\n",
    )
    if "^vision_encoder" not in source or "unexpected_keys = [x for x in unexpected_keys" not in source:
        raise RuntimeError("Text-only state-dict patch was not applied: check sam_audio/model/model.py")
    model_file.write_text(source, encoding="utf-8")

    log("Patches applied")


log("Starting SAM-Audio text-only setup. This should usually take a few minutes on a fresh Colab runtime.")

if FRESH_CLONE and REPO_DIR.exists():
    log(f"Removing existing repository: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run_step("Cloning SAM-Audio repository", ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    log(f"Repository already exists: {REPO_DIR}")

os.chdir(REPO_DIR)
log(f"Working directory: {pathlib.Path.cwd()}")
patch_sam_audio_for_colab(REPO_DIR)

pip = [sys.executable, "-m", "pip", "install", "--progress-bar", "off"]

run_step("Upgrading pip tooling", pip + ["--upgrade", "pip", "setuptools", "wheel"])
run_step(
    "Installing runtime dependencies",
    pip + [
        "numpy>=2.0,<2.1",
        "numba>=0.60",
        "einops",
        "pydub",
        "torchdiffeq",
        "torchcodec",
        "transformers>=4.54,<5",
        "huggingface_hub>=0.34,<1.0",
        "protobuf",
        "tensorboard",
        "sentencepiece",
        "accelerate",
    ],
)
run_step(
    "Installing DACVAE codec dependency",
    pip + ["git+https://github.com/facebookresearch/dacvae.git"],
)
run_step(
    "Installing local SAM-Audio package without optional upstream dependencies",
    pip + ["--no-deps", "-e", "."],
)

log("Verifying imports")
import torch
import torchaudio
from sam_audio import SAMAudio, SAMAudioProcessor

log(f"torch: {torch.__version__}")
log(f"torchaudio: {torchaudio.__version__}")
log("Setup complete. Continue with Hugging Face login.")

## 2. Hugging Face Login

If you use Colab Secrets, save your token as `HF_TOKEN`. Otherwise this cell will open the interactive login flow.

In [ ]:
#@title 2. Hugging Face Login
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    login(token=token)
    print("Logged in using Colab Secret HF_TOKEN.")
else:
    print("Enter a Hugging Face token with access to the selected model.")
    login()

## 3. Settings

In [ ]:
#@title 3. Main settings
MODEL_ID = "facebook/sam-audio-small" #@param ["facebook/sam-audio-small", "facebook/sam-audio-base", "facebook/sam-audio-large"]
DESCRIPTION = "man speaking" #@param {type:"string"}
CHUNK_SECONDS = 20 #@param {type:"slider", min:10, max:90, step:10}
AUDIO_SOURCE = "upload" #@param ["upload", "google_drive"]
DRIVE_AUDIO_PATH = "" #@param {type:"string"}
DOWNLOAD_RESIDUAL = True #@param {type:"boolean"}

DESCRIPTION = DESCRIPTION.strip().lower()
CHUNK_SECONDS = int(CHUNK_SECONDS)

if not DESCRIPTION:
    raise ValueError("DESCRIPTION cannot be empty. Examples: 'man speaking', 'drums', 'guitar'.")
if AUDIO_SOURCE == "google_drive" and not DRIVE_AUDIO_PATH.strip():
    raise ValueError("Set DRIVE_AUDIO_PATH or change AUDIO_SOURCE to 'upload'.")
if ("base" in MODEL_ID or "large" in MODEL_ID) and CHUNK_SECONDS > 30:
    print("Note: base and large models use more memory. If you get OOM errors, set CHUNK_SECONDS to 10 or 20.")

print(f"Model: {MODEL_ID}")
print(f"Prompt: {DESCRIPTION!r}")
print(f"Chunk length: {CHUNK_SECONDS}s")
print(f"Audio source: {AUDIO_SOURCE}")

## 4. Model Loading

In [ ]:
#@title 4. Load the model in lightweight mode
LOW_CPU_MEMORY_LOAD = True #@param {type:"boolean"}

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["SAM_AUDIO_COLAB_TEXT_ONLY"] = "1"

import gc
import inspect
import types
import torch
import torchaudio
try:
    from sam_audio import SAMAudio, SAMAudioProcessor
except ModuleNotFoundError as err:
    raise RuntimeError("SAM-Audio is not installed yet. Run cell 1: Set up SAM-Audio first.") from err

if "MODEL_ID" not in globals():
    MODEL_ID = "facebook/sam-audio-small"
    print(f"MODEL_ID was not set; using default: {MODEL_ID}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. In Colab enable: Runtime > Change runtime type > GPU.")

DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

for _name in ("model", "processor"):
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()

gpu_name = torch.cuda.get_device_name(0)
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
requested_low_cpu_memory_load = bool(LOW_CPU_MEMORY_LOAD)
use_low_cpu_memory_load = requested_low_cpu_memory_load and gpu_total_gb >= 24
load_map_location = "cuda" if use_low_cpu_memory_load else "cpu"
os.environ["SAM_AUDIO_COLAB_HALF_ON_LOAD"] = "1" if use_low_cpu_memory_load else "0"

print(f"GPU: {gpu_name} ({gpu_total_gb:.1f} GB VRAM)")
print(f"Checkpoint load location: {load_map_location}")
if requested_low_cpu_memory_load and not use_low_cpu_memory_load:
    print("LOW_CPU_MEMORY_LOAD was disabled automatically because this GPU has less than 24 GB VRAM. Use the small model on T4, or switch to A100/L4 for base/large.")
if use_low_cpu_memory_load and "SAM_AUDIO_COLAB_HALF_ON_LOAD" not in inspect.getsource(SAMAudio._from_pretrained):
    raise RuntimeError("Low-memory loader patch is not active. Run cell 1 again with FRESH_CLONE=True, then rerun this cell.")
if "SAM_AUDIO_COLAB_TEXT_ONLY" not in inspect.getsource(SAMAudio._from_pretrained):
    raise RuntimeError("Text-only checkpoint patch is not active. Run cell 1 again with FRESH_CLONE=True, then rerun this cell.")
if gpu_total_gb < 24 and MODEL_ID != "facebook/sam-audio-small":
    print("Warning: this runtime is below 24 GB VRAM. For the most reliable Colab run, select facebook/sam-audio-small in cell 3.")

processor = SAMAudioProcessor.from_pretrained(MODEL_ID)
model = SAMAudio.from_pretrained(
    MODEL_ID,
    visual_ranker=None,
    text_ranker=None,
    span_predictor=None,
    map_location=load_map_location,
).eval()

if next(model.parameters()).device.type != "cuda":
    model = model.to(DEVICE)

# Reduce memory: keep the model in fp16 and the audio codec in fp32 to avoid conv1d errors.
# In low-CPU-memory mode the patched loader already converts the non-codec modules before loading weights.
if not use_low_cpu_memory_load:
    model = model.half()
model.audio_codec = model.audio_codec.float()

# Streamlined mode: no reranking/spans, so these modules are not needed.
for module_name in ("visual_ranker", "text_ranker", "span_predictor"):
    if hasattr(model, module_name):
        delattr(model, module_name)

# Run the codec in fp32 and the rest of the model in fp16.
def _get_audio_features_patched(self, audios: torch.Tensor):
    audio_features = self.audio_codec(audios.float()).transpose(1, 2).half()
    return torch.cat([audio_features, audio_features], dim=2)

model._get_audio_features = types.MethodType(_get_audio_features_patched, model)

# Decode one item at a time: slower, but much more stable on T4.
_original_decode = model.audio_codec.decode

def _decode_patched(encoded_frames):
    decoded = []
    for index in range(encoded_frames.size(0)):
        decoded.append(_original_decode(encoded_frames[index:index + 1].float()))
        torch.cuda.empty_cache()
    return torch.cat(decoded, dim=0)

model.audio_codec.decode = _decode_patched

gc.collect()
torch.cuda.empty_cache()

SAMPLE_RATE = processor.audio_sampling_rate
print(f"Model loaded: {MODEL_ID}")
print(f"Output sample rate: {SAMPLE_RATE} Hz")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. Audio Input

In [ ]:
#@title 5. Load audio and create chunks
from pathlib import Path
from IPython.display import Audio, display
import math
import shutil
import torchaudio
from google.colab import files

if "AUDIO_SOURCE" not in globals():
    AUDIO_SOURCE = "upload"
    print("AUDIO_SOURCE was not set; using default: upload")
if "DRIVE_AUDIO_PATH" not in globals():
    DRIVE_AUDIO_PATH = ""
if "CHUNK_SECONDS" not in globals():
    CHUNK_SECONDS = 30
    print("CHUNK_SECONDS was not set; using default: 30")
CHUNK_SECONDS = int(CHUNK_SECONDS)

INPUT_DIR = Path("/content/sam_audio_input")
CHUNK_DIR = Path("/content/sam_audio_chunks")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

if CHUNK_DIR.exists():
    shutil.rmtree(CHUNK_DIR)
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

if AUDIO_SOURCE == "upload":
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    source_path = Path(uploaded_name)
    AUDIO_PATH = INPUT_DIR / source_path.name
    if AUDIO_PATH.exists():
        AUDIO_PATH.unlink()
    shutil.move(str(source_path), str(AUDIO_PATH))
else:
    from google.colab import drive
    drive.mount("/content/drive")
    AUDIO_PATH = Path(DRIVE_AUDIO_PATH.strip()).expanduser()
    if not AUDIO_PATH.exists():
        raise FileNotFoundError(f"File not found: {AUDIO_PATH}")

waveform, source_sr = torchaudio.load(str(AUDIO_PATH))
if waveform.numel() == 0:
    raise ValueError("The audio file appears to be empty.")

duration = waveform.shape[-1] / source_sr
print(f"File: {AUDIO_PATH}")
print(f"Duration: {duration:.1f}s | Channels: {waveform.shape[0]} | Source sample rate: {source_sr} Hz")
display(Audio(str(AUDIO_PATH)))

chunk_samples = max(1, int(CHUNK_SECONDS * source_sr))
total_samples = waveform.shape[-1]
AUDIO_CHUNKS = []

if total_samples > chunk_samples:
    num_chunks = math.ceil(total_samples / chunk_samples)
    print(f"Long audio: creating {num_chunks} chunks of about {CHUNK_SECONDS}s each.")
    for index in range(num_chunks):
        start = index * chunk_samples
        end = min(start + chunk_samples, total_samples)
        chunk_path = CHUNK_DIR / f"chunk_{index + 1:03d}.wav"
        torchaudio.save(str(chunk_path), waveform[:, start:end], source_sr)
        AUDIO_CHUNKS.append(str(chunk_path))
        print(f"  {chunk_path.name}: {start / source_sr:.1f}s - {end / source_sr:.1f}s")
else:
    AUDIO_CHUNKS = [str(AUDIO_PATH)]
    print("Short audio: no chunking needed.")

print(f"Chunks ready: {len(AUDIO_CHUNKS)}")

## 6. Separation

In [ ]:
#@title 6. Separate audio and save results
from pathlib import Path
import gc
import re
import zipfile
import torch
import torchaudio

if "DESCRIPTION" not in globals():
    DESCRIPTION = "man speaking"
    print(f"DESCRIPTION was not set; using default: {DESCRIPTION!r}")
if "processor" not in globals() or "model" not in globals():
    raise RuntimeError("Model is not loaded. Run cell 4: Load the model before separating audio.")
if "AUDIO_CHUNKS" not in globals():
    raise RuntimeError("Audio chunks are not ready. Run cell 5: Load audio and create chunks first.")
if "DEVICE" not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if "SAMPLE_RATE" not in globals():
    SAMPLE_RATE = processor.audio_sampling_rate

OUTPUT_DIR = Path("/content/sam_audio_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _slugify(text):
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
    return slug or "target"


def _as_audio_tensor(value):
    if isinstance(value, (list, tuple)):
        value = value[0]
    value = value.detach().cpu()
    while value.dim() > 2 and value.shape[0] == 1:
        value = value.squeeze(0)
    if value.dim() == 1:
        value = value.unsqueeze(0)
    if value.dim() != 2:
        raise ValueError(f"Unexpected audio shape: {tuple(value.shape)}")
    return value.float().contiguous()


def _concat_audio(paths, output_path):
    waves = []
    expected_sr = None
    for path in paths:
        wave, sr = torchaudio.load(str(path))
        if expected_sr is None:
            expected_sr = sr
        elif sr != expected_sr:
            raise ValueError("Output chunks have different sample rates.")
        if wave.dim() == 1:
            wave = wave.unsqueeze(0)
        waves.append(wave)

    channels = max(wave.shape[0] for wave in waves)
    normalized = []
    for wave in waves:
        if wave.shape[0] == channels:
            normalized.append(wave)
        elif wave.shape[0] == 1:
            normalized.append(wave.repeat(channels, 1))
        else:
            raise ValueError("Output chunks have incompatible channel counts.")

    combined = torch.cat(normalized, dim=-1)
    torchaudio.save(str(output_path), combined, expected_sr)
    return output_path


def separate_audio(description, prefix=None):
    description = description.strip().lower()
    if not description:
        raise ValueError("The description cannot be empty.")

    prefix = _slugify(prefix or description)
    target_paths = []
    residual_paths = []

    for index, chunk_path in enumerate(AUDIO_CHUNKS, start=1):
        print(f"[{index}/{len(AUDIO_CHUNKS)}] {Path(chunk_path).name} -> {description!r}")
        batch = processor(
            audios=[chunk_path],
            descriptions=[description],
        ).to(DEVICE)

        with torch.inference_mode():
            result = model.separate(
                batch,
                predict_spans=False,
                reranking_candidates=1,
            )

        target = _as_audio_tensor(result.target)
        residual = _as_audio_tensor(result.residual)

        target_path = OUTPUT_DIR / f"{prefix}_target_chunk_{index:03d}.wav"
        residual_path = OUTPUT_DIR / f"{prefix}_residual_chunk_{index:03d}.wav"
        torchaudio.save(str(target_path), target, SAMPLE_RATE)
        torchaudio.save(str(residual_path), residual, SAMPLE_RATE)
        target_paths.append(target_path)
        residual_paths.append(residual_path)

        del batch, result, target, residual
        gc.collect()
        torch.cuda.empty_cache()

    target_file = OUTPUT_DIR / f"{prefix}_target.wav"
    residual_file = OUTPUT_DIR / f"{prefix}_residual.wav"
    _concat_audio(target_paths, target_file)
    _concat_audio(residual_paths, residual_file)

    zip_file = OUTPUT_DIR / f"{prefix}_sam_audio_results.zip"
    with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(target_file, arcname=target_file.name)
        archive.write(residual_file, arcname=residual_file.name)
        for path in target_paths + residual_paths:
            archive.write(path, arcname=path.name)

    print("Separation complete.")
    print(f"Target: {target_file}")
    print(f"Residual: {residual_file}")
    print(f"Zip: {zip_file}")
    return {
        "description": description,
        "target": target_file,
        "residual": residual_file,
        "zip": zip_file,
        "target_chunks": target_paths,
        "residual_chunks": residual_paths,
    }


RESULT = separate_audio(DESCRIPTION)

## 7. Preview and Download

In [ ]:
#@title 7. Preview and download
from IPython.display import Audio, display
from google.colab import files

if "RESULT" not in globals():
    raise RuntimeError("No separation result found. Run cell 6: Separate audio and save results first.")
if "DOWNLOAD_RESIDUAL" not in globals():
    DOWNLOAD_RESIDUAL = True

print(f"Description: {RESULT['description']!r}")
print("Isolated target")
display(Audio(str(RESULT["target"])))

print("Residual")
display(Audio(str(RESULT["residual"])))

files.download(str(RESULT["target"]))
if DOWNLOAD_RESIDUAL:
    files.download(str(RESULT["residual"]))
files.download(str(RESULT["zip"]))

## 8. Optional: Multiple Prompts

Leave the field empty if you do not need this. If you enter multiple comma-separated prompts, the notebook reuses the same audio and generates one output set for each prompt.

In [ ]:
#@title 8. Optional multiple separations
MULTI_DESCRIPTIONS = "" #@param {type:"string"}

if MULTI_DESCRIPTIONS.strip():
    MULTI_RESULTS = []
    prompts = [item.strip().lower() for item in MULTI_DESCRIPTIONS.split(",") if item.strip()]
    for prompt in prompts:
        MULTI_RESULTS.append(separate_audio(prompt))
    print(f"Completed {len(MULTI_RESULTS)} multiple separations.")
else:
    print("No multiple prompts set: cell skipped.")